# Cluster Overview

This notebook queries the OCP audit SQLite datastore to present the **Cluster Overview** report — a single-row identity card per cluster covering:

- Version & Identity
- Platform & Topology
- Node Sizing
- Network Configuration
- Access Endpoints
- Update Posture

In [1]:
import os
import sys

import pandas as pd

# Shared notebook helpers (sys.path + OCP_AUDIT_DB + styling)
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from notebook_style import bootstrap, format_age_years_days, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

session, engine = bootstrap()

from schema.models import Cluster, ClusterOverview  # noqa: E402

print(f"Connected to: {engine.url}")


python: /home/vagrant/git/openshift-csv-exporter/notebook/.venv/bin/python
cwd: /home/vagrant/git/openshift-csv-exporter/notebook


Connected to: sqlite:////home/vagrant/git/openshift-csv-exporter/datastore/ocp_audit.db


## Cluster Inventory

In [2]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

9 cluster(s) in dataset


---
## Version & Identity

OCP version, Kubernetes version, cluster ID, install date, and cluster age.

In [3]:
df_version = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
df_version.insert(
    df_version.columns.get_loc("cluster_age_days") + 1,
    "cluster_age",
    df_version["cluster_age_days"].map(format_age_years_days),
)
style_table(df_version)

---
## Platform & Topology

Infrastructure platform and control-plane / infrastructure topology.

In [4]:
df_platform = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_platform)

---
## Node Sizing

Master, worker, infra, and total node counts per cluster.

In [5]:
df_nodes = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_nodes)

---
## Network Configuration

SDN type, cluster CIDRs, and service CIDRs.

In [6]:
df_network = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_network)

---
## Access Endpoints

Console URL, API server URL, and default ingress domain.

In [7]:
df_endpoints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.default_ingress_domain,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_endpoints)

---
## Update Posture

Current OCP version, update channel, update state, and count of available updates.

In [8]:
df_updates = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.update_channel,
        ClusterOverview.update_state,
        ClusterOverview.available_updates_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_updates)

---
## Full Cluster Overview

All 21 fields for every cluster in a single table.

In [9]:
df_full = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
        ClusterOverview.default_ingress_domain,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.update_channel,
        ClusterOverview.available_updates_count,
        ClusterOverview.update_state,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_full)} cluster overview(s)")
df_full.insert(
    df_full.columns.get_loc("cluster_age_days") + 1,
    "cluster_age",
    df_full["cluster_age_days"].map(format_age_years_days),
)
style_table(df_full)

9 cluster overview(s)


In [10]:
session.close()
print("Session closed.")

Session closed.
